---
# Módulo 3: Idempotencia


##  3.1: problema — Evitar duplicidad del INSERT INTO 


In [0]:
%sql

CREATE OR REPLACE TABLE propiedades.bronze.ventas 
(
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    cliente_id STRING,
    timestamp_carga TIMESTAMP
) USING DELTA;


--------------------------------------------
-- Insertamos datos iniciales
--------------------------------------------
INSERT INTO propiedades.bronze.ventas VALUES
    ('V001', '2024-01-15', 'Laptop', 1, 1200.00, 'C001', CURRENT_TIMESTAMP()),
    ('V002', '2024-01-15', 'Mouse', 2, 25.50, 'C001', CURRENT_TIMESTAMP()),
    ('V003', '2024-01-16', 'Teclado', 1, 75.00, 'C002', CURRENT_TIMESTAMP()),
    ('V004', '2024-01-16', 'Monitor', 1, 300.00, 'C003', CURRENT_TIMESTAMP()),
    ('V005', '2024-01-17', 'Laptop', 1, 1200.00, 'C004', CURRENT_TIMESTAMP());

In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.ventas_no_idempotente;

CREATE TABLE propiedades.silver.ventas_no_idempotente
(
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;


------------------------------
-- Primera ejecución
------------------------------
INSERT INTO propiedades.silver.ventas_no_idempotente
SELECT  id_venta, fecha_venta, 
        producto, cantidad, 
        precio_unitario,
        cantidad * precio_unitario as total, cliente_id

FROM propiedades.bronze.ventas;


------------------------------
-- Validacion
------------------------------
SELECT COUNT(*) as filas_primera_ejecucion 
FROM propiedades.silver.ventas_no_idempotente;

In [0]:
%sql
-- Segunda ejecución (duplica datos)

INSERT INTO propiedades.silver.ventas_no_idempotente

SELECT  id_venta, fecha_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id

FROM    propiedades.bronze.ventas;


-------------------------------------
-- Validamos
-------------------------------------
SELECT  COUNT(*) as filas_segunda_ejecucion 
FROM    propiedades.silver.ventas_no_idempotente;


-------------------------------------
-- El SUM se duplicó!
-------------------------------------

SELECT  SUM(total) as total_ventas 
FROM    propiedades.silver.ventas_no_idempotente;

## 3.2: Solución 1 — INSERT OVERWRITE
-- USada cuando no queremos conservart los datos anteriores


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.ventas_overwrite;

CREATE TABLE propiedades.silver.ventas_overwrite (
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;



INSERT OVERWRITE propiedades.silver.ventas_overwrite

SELECT  id_venta, fecha_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id

FROM    propiedades.bronze.ventas;


SELECT COUNT(*) as filas 
FROM propiedades.silver.ventas_overwrite;

In [0]:
%sql
------------------------------------------
-- Re-ejecutar: sigue siendo 5 filas
-- Al sobre escribir todos los registros eliminamos duiplicados
------------------------------------------

INSERT OVERWRITE propiedades.silver.ventas_overwrite

SELECT  id_venta, fecha_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id

FROM    propiedades.bronze.ventas;

------------------------------------------
-- validamos los registros insertados
------------------------------------------
SELECT  COUNT(*) as filas_re_ejecucion 
FROM    propiedades.silver.ventas_overwrite;

##  3.3: Eliminar duplicar registros con — MERGE


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.ventas_merge;

CREATE TABLE propiedades.silver.ventas_merge (
    id_venta STRING NOT NULL,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;


--------------------------------------------
-- Validamos si existe el registro 
--> (SI)  --> Update
--> (NO)  --> Delete 
--------------------------------------------

MERGE INTO propiedades.silver.ventas_merge AS target

USING (
    SELECT id_venta, fecha_venta, producto, 
            cantidad, precio_unitario,
            cantidad * precio_unitario as total, cliente_id

    FROM    propiedades.bronze.ventas
) AS source

ON target.id_venta = source.id_venta

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


-----------------------------------------
--validamos
-----------------------------------------
SELECT  COUNT(*) as filas_primera 
FROM    propiedades.silver.ventas_merge;

In [0]:
%sql

-- Re-Ejecutamos

--------------------------------------------
-- Validamos si existe el registro 
--> (SI)  --> Update
--> (NO)  --> Delete 
--------------------------------------------

MERGE INTO propiedades.silver.ventas_merge AS target

USING (
    SELECT id_venta, fecha_venta, producto, 
            cantidad, precio_unitario,
            cantidad * precio_unitario as total, cliente_id

    FROM    propiedades.bronze.ventas
) AS source

ON target.id_venta = source.id_venta

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


-----------------------------------------
--validamos
-----------------------------------------
SELECT  COUNT(*) as filas_primera 
FROM    propiedades.silver.ventas_merge;

In [0]:
%sql
-------------------------------------------
-- Agregar 2 nuevas ventas al bronze
-------------------------------------------

INSERT INTO propiedades.bronze.ventas
VALUES
    ('V006', '2024-01-18', 'Webcam', 1, 50.00, 'C005', CURRENT_TIMESTAMP()),
    ('V007', '2024-01-18', 'Auriculares', 1, 80.00, 'C001', CURRENT_TIMESTAMP());

-------------------------------------------
-- MERGE de nuevo: solo agrega las 2 nuevas
-------------------------------------------

MERGE INTO propiedades.silver.ventas_merge AS target
USING (
    SELECT id_venta, fecha_venta, producto, cantidad, precio_unitario,
        cantidad * precio_unitario as total, cliente_id
    FROM propiedades.bronze.ventas
) AS source

ON target.id_venta = source.id_venta

WHEN MATCHED THEN UPDATE SET *          -- Actualizamos si existen
WHEN NOT MATCHED THEN INSERT *;         -- Insertamos nuevos


-------------------------------------------
-- Validamos que no haya duplicados
-------------------------------------------
SELECT COUNT(*) as filas_con_nuevas 
FROM    propiedades.silver.ventas_merge;

## 3.4: MERGE con Hash — Idempotencia simplificada


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.silver.ventas_merge_hash;

CREATE TABLE propiedades.silver.ventas_merge_hash 
(
    row_hash STRING NOT NULL,
    id_venta STRING,
    fecha_venta DATE,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING
) USING DELTA;

-------------------------------------------
-- MErge
-- KEY llave Hash
-------------------------------------------
MERGE INTO propiedades.silver.ventas_merge_hash AS target
USING (
    SELECT 
            MD5(CONCAT_WS('|', id_venta, CAST(cantidad * precio_unitario AS STRING))) AS row_hash,
            id_venta, fecha_venta, producto, 
            cantidad, precio_unitario,
            cantidad * precio_unitario as total, cliente_id

    FROM    propiedades.bronze.ventas
) AS source

ON target.row_hash = source.row_hash        -- Usamos como KEY la llave HASH concatena 2 columnas unicas

WHEN NOT MATCHED THEN INSERT *;             -- Insertamos los nuevos regristros


-------------------------------------------
-- Validacion
-------------------------------------------
SELECT  COUNT(*) as filas, 
        COUNT(DISTINCT row_hash) as hashes_unicos

FROM    propiedades.silver.ventas_merge_hash;

Re-ejecucion
deben continuar el mismo numero de registros


In [0]:
%sql
-------------------------------------------
-- MErge
-- KEY llave Hash
-------------------------------------------
MERGE INTO propiedades.silver.ventas_merge_hash AS target
USING (
    SELECT 
            MD5(CONCAT_WS('|', id_venta, CAST(cantidad * precio_unitario AS STRING))) AS row_hash,
            id_venta, fecha_venta, producto, 
            cantidad, precio_unitario,
            cantidad * precio_unitario as total, cliente_id

    FROM    propiedades.bronze.ventas
) AS source

ON target.row_hash = source.row_hash        -- Usamos como KEY la llave HASH concatena 2 columnas unicas

WHEN NOT MATCHED THEN INSERT *;             -- Insertamos los nuevos regristros


-------------------------------------------
-- Validacion
-------------------------------------------
SELECT  COUNT(*) as filas, 
        COUNT(DISTINCT row_hash) as hashes_unicos

FROM    propiedades.silver.ventas_merge_hash;